# JSON Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. First round trip.** `dumps` makes text, `loads` parses it back — the extra letter means *string*.

In [ ]:
import json

student = {"name": "Sarah", "age": 24,
           "courses": ["math", "cs"], "active": True}

text = json.dumps(student)
print(text)                   # {"name": "Sarah", ..., "active": true}
print(type(text).__name__)    # str

back = json.loads(text)
print(back == student)        # True

**2. Lost in translation.** Tuples become lists, `True` shrinks to `true`, `None` becomes `null` — JSON knows fewer types than Python.

In [ ]:
import json

data = {"id": 7, "tags": ("ml", "llm"), "score": 91.5,
        "passed": True, "note": None}

text = json.dumps(data)
print(text)
# {"id": 7, "tags": ["ml", "llm"], "score": 91.5, "passed": true, "note": null}

back = json.loads(text)
print(back["tags"], type(back["tags"]).__name__)   # ['ml', 'llm'] list

**3. Readable Unicode.** By default non-ASCII characters escape to `\uXXXX` form; `ensure_ascii=False` keeps the real characters (the file must then be UTF-8).

In [ ]:
import json

customer = {"name": "আরিফ হাসান", "city": "Dhaka"}

print(json.dumps(customer))
# {"name": "\u0986\u09b0\u09bf\u09ab \u09b9\u09be\u09b8\u09be\u09a8", "city": "Dhaka"}

print(json.dumps(customer, ensure_ascii=False))    # humans win

## Part 2 — Practice

**4. Menu on disk.** `dump` / `load` are the file versions of `dumps` / `loads`; `indent=2` keeps the file human-readable on disk.

In [ ]:
import json
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)
menu_path = Path("sample_data/menu.json")

menu = {
    "cafe": "Dhaka Beans",
    "items": [
        {"name": "espresso", "price_bdt": 220},
        {"name": "cold brew", "price_bdt": 280},
    ],
}

with open(menu_path, "w", encoding="utf-8") as f:
    json.dump(menu, f, indent=2)

with open(menu_path, encoding="utf-8") as f:
    loaded = json.load(f)

print("total items:", len(loaded["items"]))   # total items: 2
print(loaded["items"][1])                     # {'name': 'cold brew', 'price_bdt': 280}

**5. Inside the API response.** Nested navigation is chained indexing; comprehensions collect every item from every user.

In [ ]:
response = {
    "status": "ok",
    "data": {
        "users": [
            {"id": 1, "name": "Sarah", "city": "Dhaka",
             "orders": [{"id": "A100", "total": 1250.00},
                        {"id": "A101", "total": 480.50}]},
            {"id": 2, "name": "Arif", "city": "Chattogram",
             "orders": [{"id": "B200", "total": 3200.00}]},
        ]
    },
}

users = response["data"]["users"]
print(users[0]["orders"][1]["total"])            # 480.5

for u in users:
    print(f"{u['name']} ({u['city']}): {len(u['orders'])} order(s)")

grand = sum(o["total"] for u in users for o in u["orders"])
print(f"grand total: {grand:.2f}")               # grand total: 4930.50

**6. Defensive navigation.** `.get(key, default)` turns missing fields into sensible fallbacks instead of mid-request crashes.

In [ ]:
import json

resp = json.loads('{"status": "ok", "data": {"users": []}}')

users = resp.get("data", {}).get("users", [])
print("user count:", len(users))                  # user count: 0

totals = [o["total"] for u in users for o in u.get("orders", [])]
print("grand total:", sum(totals))                # grand total: 0 - not a KeyError

## Part 3 — Challenge

**7. Rescuing custom objects.** JSON only knows six basic types — `default=` supplies a stand-in for anything else, and loaded values always come back as plain dicts/lists/strings.

In [ ]:
import json
from datetime import datetime

payload = {"user": "sarah", "logged_at": datetime(2026, 8, 26, 18, 45)}

try:
    json.dumps(payload)                    # boom - datetime unknown
except TypeError as e:
    print("TypeError:", e)

fixed = json.dumps(payload, default=str)   # teach it a fallback
print(fixed)
back = json.loads(fixed)
print(type(back["logged_at"]).__name__)    # str - the datetime became text


class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price


text = json.dumps(Product("mechanical keyboard", 2500),
                  default=lambda o: o.__dict__)
print(text)    # {"name": "mechanical keyboard", "price": 2500}
# After loading there are no Product objects - classes are forgotten.

**8. One object per line.** JSONL streams beautifully: append one line at a time, parse one line at a time.

In [ ]:
import json
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)
events = [
    {"event": "login", "user": "sarah"},
    {"event": "order", "user": "arif", "total": 480.5},
]

with open("sample_data/events.jsonl", "w", encoding="utf-8") as f:
    for e in events:
        f.write(json.dumps(e) + "\n")          # one object per line

with open("sample_data/events.jsonl", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        print(record["event"], "->", record.get("user"))
# login -> sarah
# order -> arif